# JiT-S2-VMamba — in-context **diagonal** layout, content=class

8 class tokens tied to the grid diagonal: token *e* sits just before cell (*e*, *e*) in canonical order and is **permuted together with the image** by SS2D's cross-scan (as Vim / Mamba-R's flip does for its registers): before its cell in paths 0/2, after it in the reversed paths 1/3. They enter at block 4 and stay through block 11 (`in_context_start: 4`, depth 12), then are stripped before the final layer. `in_context_len = 8`, scan `L = 64 + 8 = 72` (+12.5%) in blocks 4–11.

**Why this arm.** "Registers Matter for Pixel-Space Diffusion Transformers" finds that registers help pixel-space DiTs through extra global slots that read and write every patch. In a causal scan that depends on where the token sits. The three 8-token placements differ only in that (per block, per-cell check in `tests/test_incontext_diagonal_cpu.py`):

| placement | cells each token reads | cells each token writes to |
|---|---|---|
| prefix, K=8 | 0 (image-blind) | 64 |
| row | 0, 28, 48, 60, 64, 64, 64, 64 | 64, 64, 64, 64, 64, 60, 48, 28 |
| **diagonal (this arm)** | **64** | **64** |

One token per row **and** per column keeps the tokens evenly spaced (every 9 steps) in the row-major and the column-major scans. The diagonal itself is our choice: Mamba-R spaces its registers evenly in 1-D only, and its 2-D shape varies with the grid and register count.

**Reading the result** (same tokens, params, start block; only the placement changes): diagonal better than row and prefix K=8 → the in-context gain is partly a register effect that needs the tokens to see the image. All three tied → the gain is pure conditioning.

JiT-S2-VMamba on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.


## 1. Environment  *(Internet ON)*

In [ ]:
import os

# 1) Pin torch to 2.5.1
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = "causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
MAMBA  = "mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The diagonal layout in vmamba.py, the config and the tests must
# be on main; build_model in BOTH run_experiment.py and evaluate.py passes
# in_context_layout through unchanged.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: the layout in the model, the wiring in both entry points, and the config
!grep -q diagonal_scan_index src/models/vmamba.py \
    && grep -q in_context_layout scripts/run_experiment.py && grep -q in_context_layout scripts/evaluate.py \
    && test -f configs/tiny_imagenet/jit-s2-vmamba-incontext-diagonal-class.yaml \
    && echo "wiring OK (model + train + eval + config)" || echo "WIRING MISSING -- pull latest main"


## 3. Verify the arm  *(first session only; ~1 min)*

In [ ]:
# === Verify the diagonal arm (~2 min; first session only -- comment out on resume).
# 1) CPU proofs (reference scan, no GPU): prefix and row bit-identical to HEAD;
#    exact placement of every token and cell in all 4 paths; gather -> scan ->
#    inverse gather round trip; every token reads and writes every cell (row and
#    prefix checked per cell for comparison); params/state_dict unchanged; guards.
# 2) GPU check: CUDA kernel vs selective_scan_ref through the gather path, bf16
#    fwd+bwd with the right scan L per block, it/s + peak memory vs prefix K=8 and row.
# PYTHONPATH=. is REQUIRED: without it "from src.models.vmamba import" fails
# and the tests' try/except misreports it as a missing mamba_ssm (known issue).
# If either FAILS, STOP -- the placement is wrong and the run is meaningless.
!PYTHONPATH=. python tests/test_incontext_diagonal_cpu.py
!PYTHONPATH=. python scripts/check_incontext_diagonal_cuda.py


## 4. Download Tiny-ImageNet  *(to /tmp; comment out if you attach a dataset)*

In [ ]:
# Download Tiny-ImageNet-200 to /tmp (EPHEMERAL: keeps your saved Version small --
# only checkpoints go to /kaggle/working). Re-downloads each session (~2-3 min).
# Faster alternative: attach a Kaggle tiny-imagenet dataset and set DATA_DIR to it,
# then comment this cell out. Requires Internet ON.
import os, zipfile, glob, shutil
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
SRC      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    if not os.path.exists(ZIP):
        print("downloading tiny-imagenet-200 (~240MB)...")
        os.system(f"wget -q -O {ZIP} {SRC}")
    print("extracting...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

print("train classes:", len(glob.glob(tr+"/*")),
      "| val classes:", len(glob.glob(val+"/*")), "| DATA_DIR:", DATA_DIR)

## 5. Settings  *(edits the cloned config so train & eval agree)*

In [ ]:
# ── Settings for THIS notebook ───────────────────────────────────
CONTENT    = "class"   # in_context_content -- the diagonal layout supports class only
LENGTH     = 8            # 8 tokens on cells (e, e); must divide grid_size 8 (4 -> every 2nd cell)
START      = 4            # tokens live in blocks START..11 (depth 12)
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = "configs/tiny_imagenet/jit-s2-vmamba-incontext-diagonal-class.yaml"
OUT_DIR = "/kaggle/working/exp_incontext_diagonal_class"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). state_init stays none and ssc stays none: the arms are mutually
# exclusive (assert in the model).
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s = re.sub(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "in_context_layout": "diagonal",
                 "in_context_content": CONTENT, "in_context_len": LENGTH,
                 "in_context_start": START, "state_init": "none", "ssc": "none"})

# ── Resume across 12h sessions ─────────────────────────────────
# 1st run: RESUME_INPUT = None. Next session: Save Version, attach THIS notebook's
# previous output as an input, set RESUME_INPUT to ".../exp_incontext_diagonal_class".
RESUME_INPUT = None
import os
RESUME_CKPT = None
for c in ([os.path.join(RESUME_INPUT, "checkpoint-last.pt")] if RESUME_INPUT else []) + \
         [os.path.join(OUT_DIR, "checkpoint-last.pt")]:
    if c and os.path.exists(c):
        RESUME_CKPT = c; break
print("config :", CONFIG, "| layout: diagonal | content:", CONTENT, "| len:", LENGTH, "| blocks:", f"{START}-11")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR, "| resume:", RESUME_CKPT or "(fresh)")


## 6. Train  *(Save Version before 12h; set RESUME_INPUT next session)*

In [ ]:
# ── TRAIN ─────────────────────────────────────────────────────────────
# data_dir + the in-context knobs now live in the (edited) config; we only
# override run-specific knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}


## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
# ── EVALUATE (FID + IS + complexity). Comment out this whole cell to skip. ──
# Real reference = Tiny-IN val (10k). Uses cfg_scale 2.5 / interval [0.1,1] / EMA 1,
# matching the training config. Set --n_samples 50000 for the official-style number.
#import os
#EVAL_CKPT = f"{OUT_DIR}/checkpoint-best.pt"
#if not os.path.exists(EVAL_CKPT):
#    EVAL_CKPT = f"{OUT_DIR}/checkpoint-last.pt"
#EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
#            f"--n_samples 10000 --batch_size 200 --ema 1 "
#            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0")
#print(EVAL_CMD, "\n" + "="*70)
#!{EVAL_CMD}


## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- Keep `EPOCHS`, `cfg_scale`, and `DATA_DIR` identical to the baseline / prefix / state-init / SSC notebooks, so the only variable is the conditioning.
- Data lives in `/tmp` (not saved in the Version) to keep commits small; it re-downloads each session. To skip that, attach a Kaggle tiny-imagenet dataset and set `DATA_DIR` to it.
- Added params are only `incontext_pos_embed` (1, LENGTH, 384): 31,034,688 in total, identical to the row arm and to prefix K=8. The extra cost is sequence length in the last 8 of 12 blocks plus one gather per path; the GPU check prints it/s against both.
- FID protocol parity: same seed (42), Heun 50, cfg 2.5 / [0.1, 1.0], EMA 1, 10K samples, val reference — identical across arms; only then are the deltas attributable to the arm.
- Controls with the same 8 tokens, so the comparison is position-only: `jit-s2-vmamba-incontext-row-class` and `jit-s2-vmamba-incontext-class.yaml` with `model.in_context_len=8`.
- The prefix K-sweep is flat (K=1/16/32 → 89.20 / 89.51 / 89.44, a 0.31-point span), so read a gap of that size between placements as noise.
- Sparser variant: `LENGTH = 4` puts tokens on (0,0) (2,2) (4,4) (6,6), one every 18 scan steps; every token still reads and writes the whole grid. Use a separate `OUT_DIR`.
